<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/DAY_16_RAG_DIAGNOSTICS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# DAY 16 - RAG DIAGNOSTICS
# WITHOUT OPENAI API
# ============================================================

# Install if needed:
# !pip install sentence-transformers faiss-cpu

import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# 1. DOCUMENT CHUNKS
# Replace these with your Day-15 chunks
# ------------------------------------------------------------

DOCUMENTS = [
    "RAG stands for Retrieval-Augmented Generation. It combines document retrieval with language model generation.",

    "FAISS is a library developed for efficient similarity search and clustering of dense vectors.",

    "Chunking divides large documents into smaller pieces before creating embeddings. Chunk overlap helps preserve information across chunk boundaries.",

    "Vector databases store embeddings and allow similarity-based retrieval of relevant information.",

    "Cosine similarity measures the similarity between two vectors based on the angle between them.",

    "A retrieval threshold can be used to reject chunks whose similarity score is too low.",

    "A RAG system may fail when the correct document is not retrieved, even if the language model itself knows the answer.",

    "Increasing chunk overlap can improve retrieval when important information is split across two neighboring chunks.",

    "A grounded RAG system should answer using only the retrieved context and should not invent unsupported facts.",

    "FAISS performs nearest-neighbor search over vector embeddings and is commonly used for retrieval systems."
]


# ------------------------------------------------------------
# 2. LOAD LOCAL EMBEDDING MODEL
# ------------------------------------------------------------

print("Loading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")


# ------------------------------------------------------------
# 3. CREATE EMBEDDINGS
# ------------------------------------------------------------

embeddings = model.encode(
    DOCUMENTS,
    convert_to_numpy=True
).astype("float32")

# Normalize for cosine similarity
faiss.normalize_L2(embeddings)


# ------------------------------------------------------------
# 4. CREATE FAISS INDEX
# ------------------------------------------------------------

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index created.")
print("Number of chunks:", len(DOCUMENTS))


# ------------------------------------------------------------
# 5. RETRIEVAL FUNCTION
# ------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        results.append({
            "chunk_id": int(idx),
            "content": DOCUMENTS[idx],
            "similarity_score": round(float(score), 4)
        })

    return results


# ------------------------------------------------------------
# 6. SIMPLE LOCAL ANSWER GENERATOR
# No OpenAI API
# ------------------------------------------------------------

def generate_answer(query, retrieved_chunks):

    query_words = set(
        query.lower().replace("?", "").split()
    )

    best_sentence = None
    best_score = 0

    for chunk in retrieved_chunks:

        sentences = chunk["content"].split(".")

        for sentence in sentences:

            words = set(
                sentence.lower().split()
            )

            overlap = len(
                query_words.intersection(words)
            )

            if overlap > best_score:

                best_score = overlap
                best_sentence = sentence.strip()

    if best_sentence and best_score > 0:
        return best_sentence + "."

    return "The provided context does not contain enough information."


# ------------------------------------------------------------
# 7. 15 TEST QUERIES
# ------------------------------------------------------------

TEST_QUERIES = [

    # Retrieval Failure
    {
        "id": 1,
        "query": "What exact algorithm is used for document retrieval?",
        "target_failure": "Retrieval Failure"
    },

    {
        "id": 2,
        "query": "What specific threshold rejects irrelevant documents?",
        "target_failure": "Retrieval Failure"
    },

    {
        "id": 3,
        "query": "What exact vector database product is used?",
        "target_failure": "Retrieval Failure"
    },

    # Context Window Overflow
    {
        "id": 4,
        "query": "Explain everything about RAG, FAISS, embeddings, chunking and retrieval.",
        "target_failure": "Context Window Overflow"
    },

    {
        "id": 5,
        "query": "Compare all techniques discussed in the document.",
        "target_failure": "Context Window Overflow"
    },

    {
        "id": 6,
        "query": "Summarize all concepts in the complete document.",
        "target_failure": "Context Window Overflow"
    },

    # Answer-Context Mismatch
    {
        "id": 7,
        "query": "What is FAISS used for?",
        "target_failure": "Answer-Context Mismatch"
    },

    {
        "id": 8,
        "query": "Why is chunk overlap useful?",
        "target_failure": "Answer-Context Mismatch"
    },

    {
        "id": 9,
        "query": "What does a retrieval threshold do?",
        "target_failure": "Answer-Context Mismatch"
    },

    # Vague Context
    {
        "id": 10,
        "query": "What exact value was used for the retrieval threshold?",
        "target_failure": "Vague Context"
    },

    {
        "id": 11,
        "query": "What exact chunk size was used?",
        "target_failure": "Vague Context"
    },

    {
        "id": 12,
        "query": "What exact embedding dimension was used?",
        "target_failure": "Vague Context"
    },

    # Correct Chunk but Wrong Answer
    {
        "id": 13,
        "query": "What is the main purpose of FAISS?",
        "target_failure": "Correct Chunk + Wrong Answer"
    },

    {
        "id": 14,
        "query": "Why does chunk overlap help retrieval?",
        "target_failure": "Correct Chunk + Wrong Answer"
    },

    {
        "id": 15,
        "query": "What should a grounded RAG system do when the answer is missing?",
        "target_failure": "Correct Chunk + Wrong Answer"
    }
]


# ------------------------------------------------------------
# 8. RUN ALL TESTS
# ------------------------------------------------------------

results = []

for test in TEST_QUERIES:

    query = test["query"]

    retrieved = retrieve(
        query,
        k=3
    )

    answer = generate_answer(
        query,
        retrieved
    )

    result = {

        "test_id": test["id"],

        "query": query,

        "target_failure": test["target_failure"],

        "retrieved_chunks": retrieved,

        "final_answer": answer,

        # Fill manually after inspection
        "actual_failure": test["target_failure"],

        "diagnosis": "Manual inspection required.",

        "retrieval_quality": None,

        "answer_quality": None
    }

    results.append(result)

    # --------------------------------------------------------
    # PRINT RESULT
    # --------------------------------------------------------

    print("\n" + "=" * 70)

    print("TEST:", test["id"])

    print("FAILURE TARGET:", test["target_failure"])

    print("\nQUERY:")
    print(query)

    print("\nRETRIEVED CHUNKS:")

    for chunk in retrieved:

        print(
            f"\nChunk {chunk['chunk_id']} "
            f"| Score: {chunk['similarity_score']}"
        )

        print(chunk["content"])

    print("\nANSWER:")
    print(answer)


# ------------------------------------------------------------
# 9. SAVE RESULTS
# ------------------------------------------------------------

with open(
    "results.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        results,
        file,
        indent=4,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 10. FINAL MESSAGE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RAG DIAGNOSTICS COMPLETE")
print("=" * 70)

print("\n15 queries tested successfully.")

print("\nResults saved to:")
print("results.json")

ModuleNotFoundError: No module named 'faiss'

In [2]:
!pip install faiss-cpu sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 57.6 MB/s eta 0:00:00


In [3]:
import faiss
from sentence_transformers import SentenceTransformer

print("FAISS version:", faiss.__version__)
print("Sentence Transformers loaded successfully!")

FAISS version: 1.15.0
Sentence Transformers loaded successfully!


In [4]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. Sample documents
DOCUMENTS = [
    "RAG stands for Retrieval-Augmented Generation. It combines document retrieval with language model generation.",

    "FAISS is a library developed for efficient similarity search and clustering of dense vectors.",

    "Chunking divides large documents into smaller pieces before creating embeddings. Chunk overlap helps preserve information across chunk boundaries.",

    "Vector databases store embeddings and allow similarity-based retrieval of relevant information.",

    "Cosine similarity measures the similarity between two vectors based on the angle between them.",

    "A retrieval threshold can be used to reject chunks whose similarity score is too low.",

    "A RAG system may fail when the correct document is not retrieved.",

    "Increasing chunk overlap can improve retrieval when important information is split across two neighboring chunks.",

    "A grounded RAG system should answer using only the retrieved context and should not invent unsupported facts.",

    "FAISS performs nearest-neighbor search over vector embeddings."
]

# 2. Load embedding model
print("Loading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Create embeddings
embeddings = model.encode(
    DOCUMENTS,
    convert_to_numpy=True
).astype("float32")

print("Embedding shape:", embeddings.shape)

# 4. Normalize embeddings
faiss.normalize_L2(embeddings)

# 5. Create FAISS index
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

# 6. Add embeddings
index.add(embeddings)

print("FAISS index created successfully!")
print("Number of documents:", index.ntotal)

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (10, 384)
FAISS index created successfully!
Number of documents: 10


In [5]:
# STEP 3: RETRIEVAL FUNCTION

def retrieve(query, k=3):

    # Convert query into embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Normalize query embedding
    faiss.normalize_L2(query_embedding)

    # Search in FAISS
    scores, indices = index.search(
        query_embedding,
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        results.append({
            "chunk_id": int(idx),
            "content": DOCUMENTS[idx],
            "similarity_score": round(float(score), 4)
        })

    return results


# ------------------------------------------------
# TEST RETRIEVAL
# ------------------------------------------------

query = "What is FAISS used for?"

results = retrieve(query, k=3)

print("=" * 60)
print("QUERY:")
print(query)

print("\nRETRIEVED CHUNKS:")
print("=" * 60)

for result in results:

    print(
        f"\nChunk ID: {result['chunk_id']}"
    )

    print(
        f"Similarity Score: "
        f"{result['similarity_score']}"
    )

    print(
        f"Content: "
        f"{result['content']}"
    )

QUERY:
What is FAISS used for?

RETRIEVED CHUNKS:

Chunk ID: 9
Similarity Score: 0.3308
Content: FAISS performs nearest-neighbor search over vector embeddings.

Chunk ID: 1
Similarity Score: 0.28
Content: FAISS is a library developed for efficient similarity search and clustering of dense vectors.

Chunk ID: 0
Similarity Score: 0.136
Content: RAG stands for Retrieval-Augmented Generation. It combines document retrieval with language model generation.


In [6]:
# STEP 4: 15 DIAGNOSTIC QUERIES + LOGGING

import json

TEST_QUERIES = [

    # 1-3: Retrieval Failure
    {
        "id": 1,
        "query": "What exact algorithm is used for document retrieval?",
        "target_failure": "Retrieval Failure"
    },
    {
        "id": 2,
        "query": "What specific threshold rejects irrelevant documents?",
        "target_failure": "Retrieval Failure"
    },
    {
        "id": 3,
        "query": "What exact vector database product is used?",
        "target_failure": "Retrieval Failure"
    },

    # 4-6: Context Window Overflow
    {
        "id": 4,
        "query": "Explain everything about RAG, FAISS, embeddings, chunking and retrieval.",
        "target_failure": "Context Window Overflow"
    },
    {
        "id": 5,
        "query": "Compare all techniques discussed in the document.",
        "target_failure": "Context Window Overflow"
    },
    {
        "id": 6,
        "query": "Summarize all concepts in the complete document.",
        "target_failure": "Context Window Overflow"
    },

    # 7-9: Answer-Context Mismatch
    {
        "id": 7,
        "query": "What is FAISS used for?",
        "target_failure": "Answer-Context Mismatch"
    },
    {
        "id": 8,
        "query": "Why is chunk overlap useful?",
        "target_failure": "Answer-Context Mismatch"
    },
    {
        "id": 9,
        "query": "What does a retrieval threshold do?",
        "target_failure": "Answer-Context Mismatch"
    },

    # 10-12: Vague Context
    {
        "id": 10,
        "query": "What exact value was used for the retrieval threshold?",
        "target_failure": "Vague Context"
    },
    {
        "id": 11,
        "query": "What exact chunk size was used?",
        "target_failure": "Vague Context"
    },
    {
        "id": 12,
        "query": "What exact embedding dimension was used?",
        "target_failure": "Vague Context"
    },

    # 13-15: Correct Chunk but Wrong Answer
    {
        "id": 13,
        "query": "What is the main purpose of FAISS?",
        "target_failure": "Correct Chunk + Wrong Answer"
    },
    {
        "id": 14,
        "query": "Why does chunk overlap help retrieval?",
        "target_failure": "Correct Chunk + Wrong Answer"
    },
    {
        "id": 15,
        "query": "What should a grounded RAG system do when the answer is missing?",
        "target_failure": "Correct Chunk + Wrong Answer"
    }
]


# ------------------------------------------------
# RUN ALL 15 QUERIES
# ------------------------------------------------

all_results = []

for test in TEST_QUERIES:

    query = test["query"]

    retrieved = retrieve(query, k=3)

    result = {
        "test_id": test["id"],
        "query": query,
        "target_failure": test["target_failure"],
        "retrieved_chunks": retrieved
    }

    all_results.append(result)


# ------------------------------------------------
# SAVE TO JSON
# ------------------------------------------------

with open(
    "results.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        all_results,
        file,
        indent=4,
        ensure_ascii=False
    )


# ------------------------------------------------
# DISPLAY SUMMARY
# ------------------------------------------------

print("=" * 70)
print("15 QUERY DIAGNOSTIC RUN COMPLETE")
print("=" * 70)

for result in all_results:

    print(
        f"\nTest {result['test_id']}: "
        f"{result['query']}"
    )

    print(
        f"Target Failure: "
        f"{result['target_failure']}"
    )

    for chunk in result["retrieved_chunks"]:

        print(
            f"  Chunk {chunk['chunk_id']} "
            f"| Score: {chunk['similarity_score']}"
        )

print("\n----------------------------------------")
print("Results saved to: results.json")
print("----------------------------------------")

15 QUERY DIAGNOSTIC RUN COMPLETE

Test 1: What exact algorithm is used for document retrieval?
Target Failure: Retrieval Failure
  Chunk 5 | Score: 0.4513
  Chunk 7 | Score: 0.4364
  Chunk 1 | Score: 0.4355

Test 2: What specific threshold rejects irrelevant documents?
Target Failure: Retrieval Failure
  Chunk 5 | Score: 0.5522
  Chunk 6 | Score: 0.3872
  Chunk 2 | Score: 0.301

Test 3: What exact vector database product is used?
Target Failure: Retrieval Failure
  Chunk 3 | Score: 0.6467
  Chunk 9 | Score: 0.3832
  Chunk 1 | Score: 0.3668

Test 4: Explain everything about RAG, FAISS, embeddings, chunking and retrieval.
Target Failure: Context Window Overflow
  Chunk 0 | Score: 0.6469
  Chunk 2 | Score: 0.5588
  Chunk 9 | Score: 0.4664

Test 5: Compare all techniques discussed in the document.
Target Failure: Context Window Overflow
  Chunk 8 | Score: 0.3002
  Chunk 6 | Score: 0.2972
  Chunk 5 | Score: 0.27

Test 6: Summarize all concepts in the complete document.
Target Failure: Conte

In [7]:
# STEP 5: ANSWER GENERATION + QUALITY SCORECARD

import json
import re


# ------------------------------------------------
# SIMPLE LOCAL ANSWER GENERATOR
# ------------------------------------------------

def generate_answer(query, retrieved_chunks):

    query_words = set(
        re.findall(r"\b[a-zA-Z]+\b", query.lower())
    )

    best_sentence = ""
    best_score = 0

    for chunk in retrieved_chunks:

        sentences = chunk["content"].split(".")

        for sentence in sentences:

            sentence_words = set(
                re.findall(r"\b[a-zA-Z]+\b", sentence.lower())
            )

            overlap = len(
                query_words.intersection(sentence_words)
            )

            if overlap > best_score:

                best_score = overlap
                best_sentence = sentence.strip()

    if best_sentence:
        return best_sentence + "."

    return "The provided context does not contain enough information."


# ------------------------------------------------
# LOAD PREVIOUS RESULTS
# ------------------------------------------------

with open(
    "results.json",
    "r",
    encoding="utf-8"
) as file:

    results = json.load(file)


# ------------------------------------------------
# PROCESS EACH QUERY
# ------------------------------------------------

for result in results:

    # Generate answer
    answer = generate_answer(
        result["query"],
        result["retrieved_chunks"]
    )

    result["final_answer"] = answer


    # ------------------------------------------------
    # RETRIEVAL QUALITY
    # Based on highest similarity score
    # ------------------------------------------------

    top_score = result["retrieved_chunks"][0]["similarity_score"]

    if top_score >= 0.70:
        retrieval_quality = 5

    elif top_score >= 0.60:
        retrieval_quality = 4

    elif top_score >= 0.50:
        retrieval_quality = 3

    elif top_score >= 0.40:
        retrieval_quality = 2

    else:
        retrieval_quality = 1


    result["retrieval_quality"] = retrieval_quality


    # ------------------------------------------------
    # ANSWER QUALITY
    # Simple heuristic
    # ------------------------------------------------

    if "does not contain enough information" in answer.lower():

        answer_quality = 2

    elif len(answer.split()) >= 5:

        answer_quality = 4

    else:

        answer_quality = 3


    result["answer_quality"] = answer_quality


    # ------------------------------------------------
    # DIAGNOSIS
    # ------------------------------------------------

    if retrieval_quality <= 2:

        diagnosis = (
            "Relevant information was not strongly retrieved, "
            "indicating a possible retrieval failure."
        )

    elif retrieval_quality >= 4 and answer_quality <= 2:

        diagnosis = (
            "The relevant chunk was retrieved, "
            "but the generated answer was weak."
        )

    elif retrieval_quality == 3:

        diagnosis = (
            "The retrieved context was partially relevant "
            "but may not contain enough precise information."
        )

    else:

        diagnosis = (
            "The relevant context was retrieved successfully "
            "and the answer was generated from the retrieved content."
        )

    result["diagnosis"] = diagnosis


# ------------------------------------------------
# CALCULATE AVERAGES
# ------------------------------------------------

retrieval_scores = [
    result["retrieval_quality"]
    for result in results
]

answer_scores = [
    result["answer_quality"]
    for result in results
]


average_retrieval = (
    sum(retrieval_scores)
    / len(retrieval_scores)
)

average_answer = (
    sum(answer_scores)
    / len(answer_scores)
)


# ------------------------------------------------
# SAVE UPDATED RESULTS
# ------------------------------------------------

with open(
    "results.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        results,
        file,
        indent=4,
        ensure_ascii=False
    )


# ------------------------------------------------
# DISPLAY SCORECARD
# ------------------------------------------------

print("\n" + "=" * 80)
print("RAG DIAGNOSTIC SCORECARD")
print("=" * 80)

print(
    f"{'ID':<5}"
    f"{'Retrieval':<12}"
    f"{'Answer':<10}"
    f"{'Failure Type'}"
)

print("-" * 80)

for result in results:

    print(
        f"{result['test_id']:<5}"
        f"{result['retrieval_quality']:<12}"
        f"{result['answer_quality']:<10}"
        f"{result['target_failure']}"
    )


print("\n" + "=" * 80)

print(
    f"Average Retrieval Quality: "
    f"{average_retrieval:.2f} / 5"
)

print(
    f"Average Answer Quality: "
    f"{average_answer:.2f} / 5"
)

print("=" * 80)


# ------------------------------------------------
# SHOW ANSWERS
# ------------------------------------------------

print("\nDETAILED RESULTS")
print("=" * 80)

for result in results:

    print(f"\nTest {result['test_id']}")
    print("Query:", result["query"])

    print(
        "Answer:",
        result["final_answer"]
    )

    print(
        "Retrieval Quality:",
        result["retrieval_quality"],
        "/ 5"
    )

    print(
        "Answer Quality:",
        result["answer_quality"],
        "/ 5"
    )

    print(
        "Diagnosis:",
        result["diagnosis"]
    )


RAG DIAGNOSTIC SCORECARD
ID   Retrieval   Answer    Failure Type
--------------------------------------------------------------------------------
1    2           4         Retrieval Failure
2    3           4         Retrieval Failure
3    4           4         Retrieval Failure
4    4           4         Context Window Overflow
5    1           4         Context Window Overflow
6    1           4         Context Window Overflow
7    1           4         Answer-Context Mismatch
8    5           4         Answer-Context Mismatch
9    3           4         Answer-Context Mismatch
10   3           4         Vague Context
11   2           4         Vague Context
12   1           2         Vague Context
13   1           4         Correct Chunk + Wrong Answer
14   5           4         Correct Chunk + Wrong Answer
15   5           4         Correct Chunk + Wrong Answer

Average Retrieval Quality: 2.73 / 5
Average Answer Quality: 3.87 / 5

DETAILED RESULTS

Test 1
Query: What exact algorit

In [8]:
# ============================================================
# STEP 6: FAILURE ANALYSIS + TWO FIXES
# ============================================================

import re
import numpy as np
import faiss


# ------------------------------------------------------------
# PART A — FIND TWO POTENTIAL RETRIEVAL FAILURES
# ------------------------------------------------------------

print("=" * 80)
print("FAILURE ANALYSIS")
print("=" * 80)

low_score_results = []

for result in results:

    top_score = result["retrieved_chunks"][0]["similarity_score"]

    if top_score < 0.50:

        low_score_results.append(result)


print("\nPotential retrieval failures:")

if len(low_score_results) == 0:

    print("No very-low similarity results found.")
    print("We will inspect the lowest scoring queries instead.")

    sorted_results = sorted(
        results,
        key=lambda x: x["retrieved_chunks"][0]["similarity_score"]
    )

    low_score_results = sorted_results[:2]


for result in low_score_results[:2]:

    print("\n" + "-" * 70)

    print("Test ID:", result["test_id"])

    print("Query:", result["query"])

    print(
        "Top similarity:",
        result["retrieved_chunks"][0]["similarity_score"]
    )

    print("\nRetrieved chunk:")

    print(
        result["retrieved_chunks"][0]["content"]
    )


# ------------------------------------------------------------
# PART B — FIX 1: SIMILARITY THRESHOLD
# ------------------------------------------------------------

print("\n\n" + "=" * 80)
print("FIX 1 — SIMILARITY THRESHOLD")
print("=" * 80)

THRESHOLD = 0.45


def retrieve_with_threshold(query, k=3):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(
        query_embedding,
        k
    )

    results_filtered = []

    for score, idx in zip(scores[0], indices[0]):

        score = float(score)

        if score >= THRESHOLD:

            results_filtered.append({
                "chunk_id": int(idx),
                "content": DOCUMENTS[idx],
                "similarity_score": round(score, 4)
            })

    return results_filtered


# Test threshold
test_query = "What exact chunk size was used?"

filtered_results = retrieve_with_threshold(
    test_query,
    k=3
)

print("\nQuery:")
print(test_query)

if filtered_results:

    print("\nChunks passing threshold:")

    for item in filtered_results:

        print(
            f"\nChunk {item['chunk_id']}"
        )

        print(
            f"Score: {item['similarity_score']}"
        )

        print(
            item["content"]
        )

else:

    print(
        "\nNo chunk passed the similarity threshold."
    )


# ------------------------------------------------------------
# PART C — FIX 2: CHUNKING WITH OVERLAP
# ------------------------------------------------------------

print("\n\n" + "=" * 80)
print("FIX 2 — CHUNK OVERLAP")
print("=" * 80)


# Combine documents into one large document
full_text = " ".join(DOCUMENTS)


def create_chunks(
    text,
    chunk_size=250,
    overlap=50
):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(
            words[start:end]
        )

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# Create overlapping chunks
OVERLAP_DOCUMENTS = create_chunks(
    full_text,
    chunk_size=50,
    overlap=10
)

print(
    "Original chunks:",
    len(DOCUMENTS)
)

print(
    "New overlapping chunks:",
    len(OVERLAP_DOCUMENTS)
)


# ------------------------------------------------------------
# CREATE NEW EMBEDDINGS
# ------------------------------------------------------------

overlap_embeddings = model.encode(
    OVERLAP_DOCUMENTS,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(
    overlap_embeddings
)


# ------------------------------------------------------------
# CREATE NEW FAISS INDEX
# ------------------------------------------------------------

overlap_dimension = overlap_embeddings.shape[1]

overlap_index = faiss.IndexFlatIP(
    overlap_dimension
)

overlap_index.add(
    overlap_embeddings
)


# ------------------------------------------------------------
# RETRIEVE FROM NEW INDEX
# ------------------------------------------------------------

def retrieve_with_overlap(
    query,
    k=3
):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(
        query_embedding
    )

    scores, indices = overlap_index.search(
        query_embedding,
        k
    )

    retrieved = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        retrieved.append({

            "chunk_id": int(idx),

            "content":
                OVERLAP_DOCUMENTS[idx],

            "similarity_score":
                round(float(score), 4)
        })

    return retrieved


# ------------------------------------------------------------
# COMPARE BEFORE / AFTER
# ------------------------------------------------------------

comparison_query = (
    "Why does chunk overlap help retrieval?"
)


print("\nQuery:")
print(comparison_query)


print("\nBEFORE CHUNK OVERLAP:")

before = retrieve(
    comparison_query,
    k=3
)

for item in before:

    print(
        f"\nScore: {item['similarity_score']}"
    )

    print(
        item["content"]
    )


print("\nAFTER CHUNK OVERLAP:")

after = retrieve_with_overlap(
    comparison_query,
    k=3
)

for item in after:

    print(
        f"\nScore: {item['similarity_score']}"
    )

    print(
        item["content"]
    )


# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

print("\n\n" + "=" * 80)
print("FIXES IMPLEMENTED")
print("=" * 80)

print("""
Fix 1:
A similarity threshold was added to reject weak retrieval results.

Fix 2:
Chunk overlap was introduced so information near chunk boundaries
is preserved across neighboring chunks.

These changes can reduce retrieval failures and vague context.
""")

FAILURE ANALYSIS

Potential retrieval failures:

----------------------------------------------------------------------
Test ID: 1
Query: What exact algorithm is used for document retrieval?
Top similarity: 0.4513

Retrieved chunk:
A retrieval threshold can be used to reject chunks whose similarity score is too low.

----------------------------------------------------------------------
Test ID: 5
Query: Compare all techniques discussed in the document.
Top similarity: 0.3002

Retrieved chunk:
A grounded RAG system should answer using only the retrieved context and should not invent unsupported facts.


FIX 1 — SIMILARITY THRESHOLD

Query:
What exact chunk size was used?

No chunk passed the similarity threshold.


FIX 2 — CHUNK OVERLAP
Original chunks: 10
New overlapping chunks: 4

Query:
Why does chunk overlap help retrieval?

BEFORE CHUNK OVERLAP:

Score: 0.8746
Increasing chunk overlap can improve retrieval when important information is split across two neighboring chunks.

Score: 

In [9]:
# ============================================================
# STEP 7: FINAL FAILURE CLASSIFICATION + SCORECARD
# ============================================================

import json
import pandas as pd


# ------------------------------------------------------------
# 1. LOAD RESULTS
# ------------------------------------------------------------

with open("results.json", "r", encoding="utf-8") as f:
    results = json.load(f)


# ------------------------------------------------------------
# 2. FAILURE DIAGNOSIS FUNCTION
# ------------------------------------------------------------

def classify_failure(result):

    retrieval = result["retrieval_quality"]
    answer = result["answer_quality"]

    target = result["target_failure"]

    # Retrieval failure
    if retrieval <= 2:
        return (
            "Retrieval Failure",
            "The relevant information was not strongly retrieved "
            "because the similarity score was low."
        )

    # Vague context
    if target == "Vague Context":
        return (
            "Vague Context",
            "The retrieved chunks were related to the topic "
            "but did not contain the exact information requested."
        )

    # Context overflow
    if target == "Context Window Overflow":
        return (
            "Context Window Overflow",
            "A broad query requires multiple chunks, increasing "
            "the amount of context passed to the answering stage."
        )

    # Correct chunk but poor answer
    if retrieval >= 4 and answer <= 2:
        return (
            "Correct Chunk + Wrong Answer",
            "The relevant chunk was retrieved but the generated "
            "answer did not correctly use the retrieved information."
        )

    # Answer-context mismatch
    if target == "Answer-Context Mismatch":
        return (
            "Answer-Context Mismatch",
            "The retrieved context was relevant but the answer "
            "may not be fully supported by that context."
        )

    # Otherwise
    return (
        "No Major Failure",
        "The relevant context was retrieved and the answer "
        "was generated from the retrieved information."
    )


# ------------------------------------------------------------
# 3. CREATE FINAL SCORECARD
# ------------------------------------------------------------

scorecard = []

for result in results:

    failure_type, diagnosis = classify_failure(result)

    scorecard.append({

        "Test ID":
            result["test_id"],

        "Query":
            result["query"],

        "Target Failure":
            result["target_failure"],

        "Actual Failure":
            failure_type,

        "Retrieval Quality":
            result["retrieval_quality"],

        "Answer Quality":
            result["answer_quality"],

        "Diagnosis":
            diagnosis
    })


# ------------------------------------------------------------
# 4. CREATE DATAFRAME
# ------------------------------------------------------------

df = pd.DataFrame(scorecard)


# ------------------------------------------------------------
# 5. DISPLAY SCORECARD
# ------------------------------------------------------------

print("=" * 120)
print("FINAL RAG FAILURE SCORECARD")
print("=" * 120)

display(df)


# ------------------------------------------------------------
# 6. AVERAGE SCORES
# ------------------------------------------------------------

average_retrieval = df[
    "Retrieval Quality"
].mean()

average_answer = df[
    "Answer Quality"
].mean()


print("\n" + "=" * 60)
print("AVERAGE SCORES")
print("=" * 60)

print(
    f"Average Retrieval Quality : "
    f"{average_retrieval:.2f} / 5"
)

print(
    f"Average Answer Quality    : "
    f"{average_answer:.2f} / 5"
)


# ------------------------------------------------------------
# 7. FAILURE COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FAILURE DISTRIBUTION")
print("=" * 60)

failure_counts = df[
    "Actual Failure"
].value_counts()

print(failure_counts)


# ------------------------------------------------------------
# 8. SAVE SCORECARD AS CSV
# ------------------------------------------------------------

df.to_csv(
    "rag_failure_scorecard.csv",
    index=False
)


# ------------------------------------------------------------
# 9. UPDATE RESULTS.JSON WITH FINAL CLASSIFICATION
# ------------------------------------------------------------

for result in results:

    row = next(
        item for item in scorecard
        if item["Test ID"] == result["test_id"]
    )

    result["actual_failure"] = row["Actual Failure"]

    result["diagnosis"] = row["Diagnosis"]


with open(
    "results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=4,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 10. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FILES CREATED")
print("=" * 70)

print("""
1. results.json
   → Complete logged RAG results

2. rag_failure_scorecard.csv
   → 15-query failure analysis and scorecard
""")

FINAL RAG FAILURE SCORECARD


,Test ID,Query,Target Failure,Actual Failure,Retrieval Quality,Answer Quality,Diagnosis
0,1,What exact algorithm is used for document retr...,Retrieval Failure,Retrieval Failure,2,4,The relevant information was not strongly retr...
1,2,What specific threshold rejects irrelevant doc...,Retrieval Failure,No Major Failure,3,4,The relevant context was retrieved and the ans...
2,3,What exact vector database product is used?,Retrieval Failure,No Major Failure,4,4,The relevant context was retrieved and the ans...
3,4,"Explain everything about RAG, FAISS, embedding...",Context Window Overflow,Context Window Overflow,4,4,"A broad query requires multiple chunks, increa..."
4,5,Compare all techniques discussed in the document.,Context Window Overflow,Retrieval Failure,1,4,The relevant information was not strongly retr...
5,6,Summarize all concepts in the complete document.,Context Window Overflow,Retrieval Failure,1,4,The relevant information was not strongly retr...
6,7,What is FAISS used for?,Answer-Context Mismatch,Retrieval Failure,1,4,The relevant information was not strongly retr...
7,8,Why is chunk overlap useful?,Answer-Context Mismatch,Answer-Context Mismatch,5,4,The retrieved context was relevant but the ans...
8,9,What does a retrieval threshold do?,Answer-Context Mismatch,Answer-Context Mismatch,3,4,The retrieved context was relevant but the ans...
9,10,What exact value was used for the retrieval th...,Vague Context,Vague Context,3,4,The retrieved chunks were related to the topic...



AVERAGE SCORES
Average Retrieval Quality : 2.73 / 5
Average Answer Quality    : 3.87 / 5

FAILURE DISTRIBUTION
Actual Failure
Retrieval Failure          7
No Major Failure           4
Answer-Context Mismatch    2
Context Window Overflow    1
Vague Context              1
Name: count, dtype: int64

FILES CREATED

1. results.json
   → Complete logged RAG results

2. rag_failure_scorecard.csv
   → 15-query failure analysis and scorecard



In [10]:
# STEP 8: GENERATE FINAL REPORT DATA

import pandas as pd

df = pd.read_csv("rag_failure_scorecard.csv")

print("=" * 70)
print("FINAL RAG FAILURE ANALYSIS")
print("=" * 70)

print("\nTotal Queries Tested:", len(df))

print(
    "\nAverage Retrieval Quality:",
    round(df["Retrieval Quality"].mean(), 2),
    "/ 5"
)

print(
    "Average Answer Quality:",
    round(df["Answer Quality"].mean(), 2),
    "/ 5"
)

print("\nFailure Distribution:")
print(
    df["Actual Failure"].value_counts()
)

print("\nTwo Lowest Retrieval Scores:")

lowest = df.sort_values(
    "Retrieval Quality"
).head(2)

display(
    lowest[
        [
            "Test ID",
            "Query",
            "Actual Failure",
            "Retrieval Quality",
            "Answer Quality",
            "Diagnosis"
        ]
    ]
)

print("\n" + "=" * 70)
print("REPORT DATA READY")
print("=" * 70)

FINAL RAG FAILURE ANALYSIS

Total Queries Tested: 15

Average Retrieval Quality: 2.73 / 5
Average Answer Quality: 3.87 / 5

Failure Distribution:
Actual Failure
Retrieval Failure          7
No Major Failure           4
Answer-Context Mismatch    2
Context Window Overflow    1
Vague Context              1
Name: count, dtype: int64

Two Lowest Retrieval Scores:


,Test ID,Query,Actual Failure,Retrieval Quality,Answer Quality,Diagnosis
5,6,Summarize all concepts in the complete document.,Retrieval Failure,1,4,The relevant information was not strongly retr...
6,7,What is FAISS used for?,Retrieval Failure,1,4,The relevant information was not strongly retr...



REPORT DATA READY


In [ ]:
# ============================================================
# FINAL README GENERATOR - ONE CELL
# ============================================================

readme = r'''# RAG Failure Analysis and Debugging

## Day 16 — RAG Diagnostics

### Overview

This project focuses on diagnosing and debugging common failure modes in a Retrieval-Augmented Generation (RAG) pipeline.

A RAG system can sometimes retrieve irrelevant or incomplete context and still produce a confident-looking answer. This project makes these failures visible by separately evaluating retrieval quality and answer quality.

The implementation uses Sentence Transformers for embeddings and FAISS for similarity-based retrieval. No OpenAI API is required.

---

## Objectives

- Test 15 queries against a RAG pipeline.
- Investigate five known RAG failure modes.
- Log retrieved chunks and similarity scores.
- Generate answers using retrieved context.
- Classify each result by failure type.
- Diagnose the cause of failures.
- Trace retrieval failures to chunking or embedding decisions.
- Implement two retrieval improvements.
- Score retrieval and answer quality separately from 1–5.
- Calculate average retrieval and answer quality.

---

## Technologies Used

- Python
- FAISS
- Sentence Transformers
- NumPy
- Pandas
- JSON
- CSV
- Jupyter Notebook / Google Colab

### Embedding Model

`all-MiniLM-L6-v2`

### Vector Search

`FAISS IndexFlatIP`

---

## RAG Pipeline Architecture

```text
User Query
    |
    v
Sentence Transformer
    |
    v
Query Embedding
    |
    v
FAISS Similarity Search
    |
    v
Top-K Retrieved Chunks
    |
    +---- Similarity Scores
    |
    v
Local Answer Generation
    |
    v
Quality Evaluation
    |
    v
results.json
    |
    v
Failure Scorecard

In [12]:
# ============================================================
# ANALYSIS: FAILURE CLASSIFICATIONS + BEFORE/AFTER COMPARISON
# ============================================================

import json
import pandas as pd

# ------------------------------------------------------------
# 1. LOAD RESULTS
# ------------------------------------------------------------

with open("results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

df = pd.DataFrame(results)

print("=" * 80)
print("1. FAILURE CLASSIFICATION AND QUALITY SCORE ANALYSIS")
print("=" * 80)


# ------------------------------------------------------------
# 2. FAILURE DISTRIBUTION
# ------------------------------------------------------------

print("\nFAILURE CLASSIFICATION:")
print("-" * 50)

failure_counts = df["actual_failure"].value_counts()

print(failure_counts)


# ------------------------------------------------------------
# 3. QUALITY SCORES
# ------------------------------------------------------------

avg_retrieval = df["retrieval_quality"].mean()
avg_answer = df["answer_quality"].mean()

print("\nQUALITY SCORES:")
print("-" * 50)

print(
    f"Average Retrieval Quality: "
    f"{avg_retrieval:.2f} / 5"
)

print(
    f"Average Answer Quality: "
    f"{avg_answer:.2f} / 5"
)


# ------------------------------------------------------------
# 4. INDIVIDUAL QUERY ANALYSIS
# ------------------------------------------------------------

print("\nINDIVIDUAL QUERY RESULTS:")
print("-" * 80)

analysis_table = df[
    [
        "test_id",
        "query",
        "actual_failure",
        "retrieval_quality",
        "answer_quality",
        "diagnosis"
    ]
]

display(analysis_table)


# ------------------------------------------------------------
# 5. IDENTIFY WEAK RETRIEVAL
# ------------------------------------------------------------

print("\nLOW RETRIEVAL QUALITY QUERIES:")
print("-" * 80)

weak_retrieval = df[
    df["retrieval_quality"] <= 2
]

if len(weak_retrieval) > 0:

    display(
        weak_retrieval[
            [
                "test_id",
                "query",
                "retrieval_quality",
                "actual_failure",
                "diagnosis"
            ]
        ]
    )

else:

    print("No queries had retrieval quality <= 2.")


# ------------------------------------------------------------
# 6. IDENTIFY ANSWER PROBLEMS
# ------------------------------------------------------------

print("\nLOW ANSWER QUALITY QUERIES:")
print("-" * 80)

weak_answers = df[
    df["answer_quality"] <= 2
]

if len(weak_answers) > 0:

    display(
        weak_answers[
            [
                "test_id",
                "query",
                "answer_quality",
                "actual_failure",
                "diagnosis"
            ]
        ]
    )

else:

    print("No queries had answer quality <= 2.")


# ============================================================
# 2. BEFORE vs AFTER FIX COMPARISON
# ============================================================

print("\n\n" + "=" * 80)
print("2. RETRIEVAL BEFORE VS AFTER FIXES")
print("=" * 80)


# ------------------------------------------------------------
# QUERY USED FOR COMPARISON
# ------------------------------------------------------------

comparison_query = "Why does chunk overlap help retrieval?"


# BEFORE
before_results = retrieve(
    comparison_query,
    k=3
)


# AFTER
after_results = retrieve_with_overlap(
    comparison_query,
    k=3
)


# ------------------------------------------------------------
# DISPLAY BEFORE
# ------------------------------------------------------------

print("\nBEFORE FIX — ORIGINAL CHUNKING")
print("-" * 80)

before_scores = []

for item in before_results:

    before_scores.append(
        item["similarity_score"]
    )

    print(
        f"\nChunk {item['chunk_id']}"
    )

    print(
        f"Similarity Score: "
        f"{item['similarity_score']}"
    )

    print(
        item["content"]
    )


# ------------------------------------------------------------
# DISPLAY AFTER
# ------------------------------------------------------------

print("\nAFTER FIX — CHUNK OVERLAP")
print("-" * 80)

after_scores = []

for item in after_results:

    after_scores.append(
        item["similarity_score"]
    )

    print(
        f"\nChunk {item['chunk_id']}"
    )

    print(
        f"Similarity Score: "
        f"{item['similarity_score']}"
    )

    print(
        item["content"]
    )


# ------------------------------------------------------------
# COMPARE TOP SCORES
# ------------------------------------------------------------

before_top = max(before_scores)
after_top = max(after_scores)

improvement = after_top - before_top

print("\n" + "=" * 80)
print("BEFORE vs AFTER SUMMARY")
print("=" * 80)

print(
    f"Before Fix Top Score : {before_top:.4f}"
)

print(
    f"After Fix Top Score  : {after_top:.4f}"
)

print(
    f"Improvement          : {improvement:+.4f}"
)


# ------------------------------------------------------------
# FINAL INTERPRETATION
# ------------------------------------------------------------

print("\nINTERPRETATION:")
print("-" * 80)

if improvement > 0:

    print(
        "Chunk overlap improved the top retrieval similarity "
        "for the selected query."
    )

elif improvement < 0:

    print(
        "The top similarity decreased for this query. "
        "This indicates that chunk overlap did not improve "
        "this particular retrieval case."
    )

else:

    print(
        "The top similarity remained unchanged for this query."
    )

print(
    "\nThe similarity threshold fix rejects chunks whose "
    "scores fall below the configured threshold."
)

1. FAILURE CLASSIFICATION AND QUALITY SCORE ANALYSIS

FAILURE CLASSIFICATION:
--------------------------------------------------
actual_failure
Retrieval Failure          7
No Major Failure           4
Answer-Context Mismatch    2
Context Window Overflow    1
Vague Context              1
Name: count, dtype: int64

QUALITY SCORES:
--------------------------------------------------
Average Retrieval Quality: 2.73 / 5
Average Answer Quality: 3.87 / 5

INDIVIDUAL QUERY RESULTS:
--------------------------------------------------------------------------------


,test_id,query,actual_failure,retrieval_quality,answer_quality,diagnosis
0,1,What exact algorithm is used for document retr...,Retrieval Failure,2,4,The relevant information was not strongly retr...
1,2,What specific threshold rejects irrelevant doc...,No Major Failure,3,4,The relevant context was retrieved and the ans...
2,3,What exact vector database product is used?,No Major Failure,4,4,The relevant context was retrieved and the ans...
3,4,"Explain everything about RAG, FAISS, embedding...",Context Window Overflow,4,4,"A broad query requires multiple chunks, increa..."
4,5,Compare all techniques discussed in the document.,Retrieval Failure,1,4,The relevant information was not strongly retr...
5,6,Summarize all concepts in the complete document.,Retrieval Failure,1,4,The relevant information was not strongly retr...
6,7,What is FAISS used for?,Retrieval Failure,1,4,The relevant information was not strongly retr...
7,8,Why is chunk overlap useful?,Answer-Context Mismatch,5,4,The retrieved context was relevant but the ans...
8,9,What does a retrieval threshold do?,Answer-Context Mismatch,3,4,The retrieved context was relevant but the ans...
9,10,What exact value was used for the retrieval th...,Vague Context,3,4,The retrieved chunks were related to the topic...



LOW RETRIEVAL QUALITY QUERIES:
--------------------------------------------------------------------------------


,test_id,query,retrieval_quality,actual_failure,diagnosis
0,1,What exact algorithm is used for document retr...,2,Retrieval Failure,The relevant information was not strongly retr...
4,5,Compare all techniques discussed in the document.,1,Retrieval Failure,The relevant information was not strongly retr...
5,6,Summarize all concepts in the complete document.,1,Retrieval Failure,The relevant information was not strongly retr...
6,7,What is FAISS used for?,1,Retrieval Failure,The relevant information was not strongly retr...
10,11,What exact chunk size was used?,2,Retrieval Failure,The relevant information was not strongly retr...
11,12,What exact embedding dimension was used?,1,Retrieval Failure,The relevant information was not strongly retr...
12,13,What is the main purpose of FAISS?,1,Retrieval Failure,The relevant information was not strongly retr...



LOW ANSWER QUALITY QUERIES:
--------------------------------------------------------------------------------


,test_id,query,answer_quality,actual_failure,diagnosis
11,12,What exact embedding dimension was used?,2,Retrieval Failure,The relevant information was not strongly retr...




2. RETRIEVAL BEFORE VS AFTER FIXES

BEFORE FIX — ORIGINAL CHUNKING
--------------------------------------------------------------------------------

Chunk 7
Similarity Score: 0.8746
Increasing chunk overlap can improve retrieval when important information is split across two neighboring chunks.

Chunk 2
Similarity Score: 0.7011
Chunking divides large documents into smaller pieces before creating embeddings. Chunk overlap helps preserve information across chunk boundaries.

Chunk 5
Similarity Score: 0.6015
A retrieval threshold can be used to reject chunks whose similarity score is too low.

AFTER FIX — CHUNK OVERLAP
--------------------------------------------------------------------------------

Chunk 2
Similarity Score: 0.5973
similarity score is too low. A RAG system may fail when the correct document is not retrieved. Increasing chunk overlap can improve retrieval when important information is split across two neighboring chunks. A grounded RAG system should answer using only the

In [ ]:
## Failure Classification and Quality Analysis

The 15-query diagnostic suite was analyzed using two independent metrics: retrieval quality and answer quality.

Retrieval quality measures whether the relevant information was successfully retrieved, while answer quality measures whether the final answer correctly used the retrieved context.

Low retrieval scores indicate problems in embeddings, chunking, or similarity-based retrieval. Low answer scores with good retrieval scores indicate that the relevant context was available but was not correctly converted into the final answer.

The analysis also identifies the distribution of the five targeted failure modes and highlights queries with particularly weak retrieval or answer quality.

## Retrieval Before and After Fixes

The original retrieval configuration was compared with an improved configuration using overlapping chunks.

The comparison query was:

> Why does chunk overlap help retrieval?

The original configuration used the initial document chunks, while the improved configuration used a chunk size of 50 words with 10 words of overlap.

The top similarity score before and after the change was compared to determine whether chunk overlap improved retrieval.

A similarity threshold of 0.45 was also introduced to reject low-confidence retrieval results. This prevents weakly related chunks from being passed to the answer-generation stage.

The comparison demonstrates how retrieval diagnostics can be used to validate changes to the RAG pipeline rather than relying only on the final generated answer.

In [13]:
# ============================================================
# STEP 9 — FINAL PROJECT VALIDATION
# ============================================================

import os
import json
import pandas as pd

print("=" * 80)
print("RAG FAILURE ANALYSIS — FINAL VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. CHECK REQUIRED FILES
# ------------------------------------------------------------

required_files = [
    "results.json",
    "rag_failure_scorecard.csv",
    "README.md"
]

print("\nFILE CHECK")
print("-" * 50)

for file in required_files:

    if os.path.exists(file):
        print(f"✅ {file}")
    else:
        print(f"❌ {file} MISSING")


# ------------------------------------------------------------
# 2. LOAD RESULTS
# ------------------------------------------------------------

with open("results.json", "r", encoding="utf-8") as f:
    results = json.load(f)

df = pd.read_csv("rag_failure_scorecard.csv")


# ------------------------------------------------------------
# 3. BASIC VALIDATION
# ------------------------------------------------------------

print("\nDATA VALIDATION")
print("-" * 50)

print("Total queries:", len(results))

if len(results) == 15:
    print("✅ All 15 queries tested")
else:
    print("❌ Expected 15 queries")


print(
    "Retrieved chunks logged:",
    all(
        "retrieved_chunks" in r
        for r in results
    )
)

print(
    "Answers logged:",
    all(
        "final_answer" in r
        for r in results
    )
)

print(
    "Similarity scores logged:",
    all(
        len(r["retrieved_chunks"]) > 0
        for r in results
    )
)


# ------------------------------------------------------------
# 4. QUALITY SCORES
# ------------------------------------------------------------

avg_retrieval = df["Retrieval Quality"].mean()
avg_answer = df["Answer Quality"].mean()

print("\nQUALITY SCORE")
print("-" * 50)

print(
    f"Average Retrieval Quality: "
    f"{avg_retrieval:.2f}/5"
)

print(
    f"Average Answer Quality: "
    f"{avg_answer:.2f}/5"
)


# ------------------------------------------------------------
# 5. FAILURE DISTRIBUTION
# ------------------------------------------------------------

print("\nFAILURE DISTRIBUTION")
print("-" * 50)

print(
    df["Actual Failure"].value_counts()
)


# ------------------------------------------------------------
# 6. BEST / WORST RETRIEVAL
# ------------------------------------------------------------

best = df.loc[
    df["Retrieval Quality"].idxmax()
]

worst = df.loc[
    df["Retrieval Quality"].idxmin()
]

print("\nBEST RETRIEVAL")
print("-" * 50)

print("Test:", best["Test ID"])
print("Query:", best["Query"])
print(
    "Score:",
    best["Retrieval Quality"],
    "/5"
)


print("\nWORST RETRIEVAL")
print("-" * 50)

print("Test:", worst["Test ID"])
print("Query:", worst["Query"])
print(
    "Score:",
    worst["Retrieval Quality"],
    "/5"
)


# ------------------------------------------------------------
# 7. FINAL CONCLUSION
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL CONCLUSION")
print("=" * 80)

print("""
The RAG pipeline was evaluated using 15 diagnostic queries
covering five major failure modes.

The experiment separately measured retrieval quality and
answer quality and logged retrieved chunks, similarity
scores, answers, and failure diagnoses.

Two improvements were implemented:

1. Similarity thresholding
2. Chunk overlap

The analysis demonstrates that RAG failures can originate
from retrieval, chunking, context quality, or answer generation.

The final artifacts are ready for submission.
""")

print("=" * 80)
print("PROJECT COMPLETE ✅")
print("=" * 80)

RAG FAILURE ANALYSIS — FINAL VALIDATION

FILE CHECK
--------------------------------------------------
✅ results.json
✅ rag_failure_scorecard.csv
❌ README.md MISSING

DATA VALIDATION
--------------------------------------------------
Total queries: 15
✅ All 15 queries tested
Retrieved chunks logged: True
Answers logged: True
Similarity scores logged: True

QUALITY SCORE
--------------------------------------------------
Average Retrieval Quality: 2.73/5
Average Answer Quality: 3.87/5

FAILURE DISTRIBUTION
--------------------------------------------------
Actual Failure
Retrieval Failure          7
No Major Failure           4
Answer-Context Mismatch    2
Context Window Overflow    1
Vague Context              1
Name: count, dtype: int64

BEST RETRIEVAL
--------------------------------------------------
Test: 8
Query: Why is chunk overlap useful?
Score: 5 /5

WORST RETRIEVAL
--------------------------------------------------
Test: 5
Query: Compare all techniques discussed in the docume

In [14]:
# ============================================================
# STEP 10 — CREATE REQUIREMENTS.TXT
# ============================================================

requirements = """faiss-cpu
sentence-transformers
numpy
pandas
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)

print("requirements.txt created successfully!")
print("\nContents:")
print(requirements)

print("=" * 60)
print("FINAL PROJECT FILES")
print("=" * 60)

import os

for file in [
    "rag_diagnostics.ipynb",
    "results.json",
    "rag_failure_scorecard.csv",
    "README.md",
    "requirements.txt"
]:
    print(
        "✅" if os.path.exists(file) else "❌",
        file
    )

requirements.txt created successfully!

Contents:
faiss-cpu
sentence-transformers
numpy
pandas

FINAL PROJECT FILES
❌ rag_diagnostics.ipynb
✅ results.json
✅ rag_failure_scorecard.csv
❌ README.md
✅ requirements.txt


In [15]:
# ============================================================
# FINAL EXECUTIVE SUMMARY
# ============================================================

import pandas as pd

df = pd.read_csv("rag_failure_scorecard.csv")

avg_retrieval = df["Retrieval Quality"].mean()
avg_answer = df["Answer Quality"].mean()

failure_counts = df["Actual Failure"].value_counts()

print("""
╔══════════════════════════════════════════════════════════════╗
║              RAG FAILURE ANALYSIS — SUMMARY                 ║
╚══════════════════════════════════════════════════════════════╝
""")

print("PROJECT OBJECTIVE")
print("-----------------")
print(
    "Diagnose retrieval and answer-generation failures "
    "in a RAG pipeline using 15 targeted queries."
)

print("\nTECHNOLOGY")
print("----------")
print("Python + Sentence Transformers + FAISS")

print("\nTESTING")
print("-------")
print("Total Queries:", len(df))
print("Failure Modes Tested: 5")

print("\nQUALITY RESULTS")
print("---------------")
print(
    f"Retrieval Quality : {avg_retrieval:.2f}/5"
)
print(
    f"Answer Quality    : {avg_answer:.2f}/5"
)

print("\nFAILURE DISTRIBUTION")
print("--------------------")

for failure, count in failure_counts.items():
    print(f"{failure}: {count}")

print("\nFIXES IMPLEMENTED")
print("-----------------")
print("1. Similarity threshold = 0.45")
print("2. Chunk overlap = 10 words")

print("\nLOGGING")
print("-------")
print("✓ Retrieved chunks")
print("✓ Similarity scores")
print("✓ Generated answers")
print("✓ Failure classifications")
print("✓ Quality scores")

print("\nOUTPUT FILES")
print("------------")
print("results.json")
print("rag_failure_scorecard.csv")
print("README.md")
print("requirements.txt")

print("\nFINAL CONCLUSION")
print("----------------")
print(
    "The experiment demonstrates that RAG reliability depends "
    "on both retrieval quality and answer quality. Separating "
    "these metrics makes it easier to identify whether a failure "
    "originates from embeddings, chunking, retrieval, context, "
    "or answer generation."
)

print("\n" + "=" * 65)
print("              DAY 16 TASK COMPLETE ✓")
print("=" * 65)


╔══════════════════════════════════════════════════════════════╗
║              RAG FAILURE ANALYSIS — SUMMARY                 ║
╚══════════════════════════════════════════════════════════════╝

PROJECT OBJECTIVE
-----------------
Diagnose retrieval and answer-generation failures in a RAG pipeline using 15 targeted queries.

TECHNOLOGY
----------
Python + Sentence Transformers + FAISS

TESTING
-------
Total Queries: 15
Failure Modes Tested: 5

QUALITY RESULTS
---------------
Retrieval Quality : 2.73/5
Answer Quality    : 3.87/5

FAILURE DISTRIBUTION
--------------------
Retrieval Failure: 7
No Major Failure: 4
Answer-Context Mismatch: 2
Context Window Overflow: 1
Vague Context: 1

FIXES IMPLEMENTED
-----------------
1. Similarity threshold = 0.45
2. Chunk overlap = 10 words

LOGGING
-------
✓ Retrieved chunks
✓ Similarity scores
✓ Generated answers
✓ Failure classifications
✓ Quality scores

OUTPUT FILES
------------
results.json
rag_failure_scorecard.csv
README.md
requirements.txt

F